# Demo Trực quan hóa Dự báo Bitcoin Chuỗi Thời gian (BTCUSDT 15-phút)

Notebook này đóng vai trò là một **Interactive Demo** trực quan sinh động, cho phép tải các checkpoint đã huấn luyện từ cả 4 thí nghiệm (mô hình dự báo 16 bước - 4 giờ) và so sánh trực tiếp kết quả dự báo của chúng với giá thực tế (**Ground Truth**) trên tập kiểm thử độc lập.

### 4 Biến thể Mô hình được Đối chiếu:
1. **`ori` (Original CALF)**: Cấu trúc Cross-Modal Alignment nguyên bản.
2. **`dropAttn_keepWE`**: Mô hình lược bỏ Cross-Attention nhưng giữ lại Word Token Embeddings (WTE).
3. **`llm_to_attn`**: Chưng cất tri thức ngôn ngữ sang cơ chế Cross-Attention tự thiết kế.
4. **`llm_to_trsf`**: Chưng cất tri thức ngôn ngữ sang cấu trúc mạng Transformer tự thiết kế.

## Bước 1: Khai báo Thư viện & Thiết lập Thiết bị (CPU/GPU)

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Import cấu trúc mô hình và bộ nạp dữ liệu từ mã nguồn dự án
from models import GPT4TS
from data_provider.data_loader import Dataset_Custom
from utils.torch_compat import load_state_dict_checkpoint

# Thiết lập seed để đảm bảo tính tái lập kết quả
random.seed(2021)
np.random.seed(2021)
torch.manual_seed(2021)

# Cấu hình phong cách đồ thị
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (15, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

# Tự động phát hiện GPU hoặc CPU fallback
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Đang sử dụng thiết bị:", device)

c:\Users\USER\anaconda3\envs\llm_ts\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang sử dụng thiết bị: cpu


## Bước 2: Thiết lập Tham số Cấu hình Mô hình (Dummy Config)

In [2]:
class DummyConfigs:
    def __init__(self, model_id, pred_len=16):
        self.pred_len = pred_len
        self.task_name = 'long_term_forecast'
        self.log_fine_name = 'demo.txt'
        self.model_id = model_id
        self.gpt_layers = 6
        self.word_embedding_path = 'wte_pca_500.pt'
        self.d_model = 768
        self.seq_len = 96
        self.enc_in = 7
        self.dec_in = 7
        self.c_out = 7
        self.features = 'M'
        self.target = 'close'
        self.embed = 'timeF'
        self.freq = '15min'

## Bước 3: Nạp checkpoint của cả 4 Mô hình (pred_len = 16)

In [3]:
ckpt_root = './checkpoints'

# Đường dẫn thư mục chứa checkpoint cho 4 mô hình
model_paths = {
    'ori': 'long_term_forecast_BTCUSDT_96_16_ori_GPT4TS_custom_ftM_sl96_ll0_pl16_dm768_nh8_el2_dl1_df768_fc1_ebtimeF_dtTrue_test_gpt6_0',
    'dropAttn_keepWE': 'long_term_forecast_BTCUSDT_96_16_dropAttn_keepWE_GPT4TS_custom_ftM_sl96_ll0_pl16_dm768_nh8_el2_dl1_df768_fc1_ebtimeF_dtTrue_test_gpt6_0',
    'llm_to_attn': 'long_term_forecast_BTCUSDT_96_16_llm_to_attn_GPT4TS_custom_ftM_sl96_ll0_pl16_dm768_nh8_el2_dl1_df768_fc1_ebtimeF_dtTrue_test_gpt6_0',
    'llm_to_trsf': 'long_term_forecast_BTCUSDT_96_16_llm_to_trsf_GPT4TS_custom_ftM_sl96_ll0_pl16_dm768_nh8_el2_dl1_df768_fc1_ebtimeF_dtTrue_test_gpt6_0'
}

models = {}
for name, folder in model_paths.items():
    ckpt_path = os.path.join(ckpt_root, folder, 'checkpoint.pth')
    if not os.path.exists(ckpt_path):
        print(f"[CẢNH BÁO]: Không tìm thấy checkpoint tại {ckpt_path}")
        continue
    
    model_id = f"BTCUSDT_96_16_{name}"
    cfg = DummyConfigs(model_id=model_id, pred_len=16)
    
    # Khởi tạo mô hình
    model = GPT4TS.Model(cfg, device).float()
    
    # Nạp trọng số trọng lượng đã huấn luyện
    state_dict = load_state_dict_checkpoint(ckpt_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    models[name] = model
    print(f"Đã nạp thành công mô hình: {name}")

model_id  BTCUSDT_96_16_ori


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy.core.multiarray._reconstruct])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch

## Bước 4: Khởi tạo Bộ Dữ liệu Kiểm thử (Test Dataset Split)

In [ ]:
# Nạp phần dữ liệu kiểm thử độc lập (chiếm 20% cuối chuỗi thời gian của bộ dữ liệu đã làm sạch)
dataset = Dataset_Custom(
    root_path='./dataset/clean/',
    data_path='BTCUSDT_15m_calf.csv',
    flag='test',
    size=[96, 0, 16],
    features='M',
    target='close',
    scale=True,
    timeenc=1,
    freq='15min'
)
print(f"Đã tải xong tập dữ liệu kiểm thử. Tổng số mẫu có sẵn: {len(dataset)}")

## Bước 5: Định nghĩa Hàm Dự báo & Trực quan hóa So sánh kết quả

In [ ]:
def run_demo_forecast(index):
    # 1. Trích xuất mẫu dữ liệu tại index đã chọn
    seq_x, seq_y, seq_x_mark, seq_y_mark = dataset[index]
    
    # 2. Thêm batch dimension và đưa lên thiết bị phần cứng
    x = torch.tensor(seq_x).float().unsqueeze(0).to(device)  # (1, 96, 7)
    x_mark = torch.tensor(seq_x_mark).float().unsqueeze(0).to(device)  # (1, 96, 4)
    
    # 3. Chạy dự báo thông qua 4 mô hình độc lập
    preds = {}
    with torch.no_grad():
        for name, model in models.items():
            out = model(x)
            out_ensemble = out['outputs_time']
            pred = out_ensemble[0, -16:, :].cpu().numpy()  # Lấy 16 bước dự báo (16, 7)
            preds[name] = pred
            
    # 4. Giải chuẩn hóa (Inverse transform) về tỷ giá USD Bitcoin thực tế
    history_original = dataset.inverse_transform(seq_x)  # (96, 7)
    true_original = dataset.inverse_transform(seq_y)  # (16, 7)
    
    preds_original = {}
    for name, p in preds.items():
        preds_original[name] = dataset.inverse_transform(p)
        
    # Chỉ số cột của 'close' (Giá đóng cửa) là 6
    target_idx = 6
    
    history_prices = history_original[:, target_idx]
    true_prices = true_original[:, target_idx]
    
    # Vẽ biểu đồ đối sánh
    plt.figure(figsize=(15, 7))
    
    # Vẽ lịch sử (lấy 48 bước gần nhất = 12 giờ trước ranh giới dự báo cho dễ quan sát)
    hist_len_to_plot = 48
    x_hist = np.arange(-hist_len_to_plot, 0)
    plt.plot(x_hist, history_prices[-hist_len_to_plot:], color='#2c3e50', label='Giá lịch sử (BTCUSD)', linewidth=2.2)
    
    # Vẽ giá thực tế Ground Truth (Đường màu đen liền nét)
    x_pred = np.arange(0, 16)
    plt.plot(np.insert(x_pred, 0, -1), np.insert(true_prices, 0, history_prices[-1]), 
             color='black', label='Thực tế (Ground Truth)', linewidth=2.8, linestyle='-')
    
    # Bảng màu tương phản cho các mô hình
    colors = {
        'ori': '#e74c3c',            # Đỏ đậm
        'dropAttn_keepWE': '#3498db', # Xanh biển
        'llm_to_attn': '#2ecc71',     # Xanh lá
        'llm_to_trsf': '#f1c40f'      # Vàng hổ phách
    }
    
    # Vẽ dự báo của từng mô hình cùng chỉ số MAE / MSE thực tế trên đoạn này
    for name, p_orig in preds_original.items():
        p_prices = p_orig[:, target_idx]
        mae = np.mean(np.abs(p_prices - true_prices))
        mse = np.mean((p_prices - true_prices) ** 2)
        
        plt.plot(np.insert(x_pred, 0, -1), np.insert(p_prices, 0, history_prices[-1]), 
                 color=colors[name], label=f"{name} (MAE: {mae:.2f}, MSE: {mse:.2f})", 
                 linewidth=2.0, linestyle='--')
        
    # Đường phân tách ranh giới dự báo
    plt.axvline(x=-1, color='gray', linestyle=':', linewidth=1.5, label='Ranh giới Dự báo')
    
    plt.title(f"Interactive Forecast Demo - Bitcoin Tỷ giá USD (Index kiểm thử: {index})", fontsize=14, fontweight='bold', color='#2c3e50')
    plt.xlabel("Mốc thời gian (Mỗi mốc = 15 phút | Dự báo 4 giờ tiếp theo)", fontsize=12)
    plt.ylabel("Tỷ giá Bitcoin (USD)", fontsize=12)
    plt.legend(loc='upper left', frameon=True, fontsize=10.5, facecolor='white', edgecolor='#e2e8f0')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

## Bước 6: Thử nghiệm Dự báo một vị trí Cố định trên tập Kiểm thử

In [ ]:
# Bạn có thể thay đổi chỉ số (Index) dưới đây từ 0 đến khoảng 14000 để kiểm tra các thời điểm khác nhau!
selected_test_index = 320
run_demo_forecast(selected_test_index)

## Bước 7: Dự báo Ngẫu nhiên trên tập Kiểm thử (Xem các tình huống thị trường ngẫu nhiên)

In [ ]:
random_test_index = random.randint(0, len(dataset) - 1)
print(f"--- Đang chọn ngẫu nhiên thời điểm kiểm thử tại index: {random_test_index} ---")
run_demo_forecast(random_test_index)